# 07 — Walk-forward model vs market backtest

This notebook is the next validation step after building football features and market joins.

Why it is needed:
- a large football-plus-market model can overfit on a small odds dataset;
- the market is a very strong probability baseline;
- therefore we run a chronological walk-forward evaluation.

What it does:
1. For each market match, train the football model only on matches before that date.
2. Generate football probabilities.
3. Compare them with T-24h market probabilities.
4. Search for a simple market-plus-football blend.
5. Build a paper-only ROI and closing-line-value backtest.

API calls: 0.

Output:
- `data/processed/07_walkforward_market_backtest_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import re
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
OUT_DIR = PROCESSED_DIR / "07_walkforward_market_backtest"

OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def make_logit_pipeline(C=0.5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            C=C,
            random_state=SEED,
        )),
    ])

def multiclass_brier(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    one_hot = np.zeros_like(proba, dtype=float)
    one_hot[np.arange(len(y_true)), y_true] = 1.0
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))

def evaluate_probs(name, y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    pred = np.argmax(proba, axis=1)
    return {
        "model": name,
        "n": len(y_true),
        "accuracy": float(accuracy_score(y_true, pred)),
        "log_loss": float(log_loss(y_true, proba, labels=[0, 1, 2])),
        "brier": multiclass_brier(y_true, proba),
    }

print("Setup OK.")


In [ ]:
# Cell 2. Load required files.

training_path = PROCESSED_DIR / "training_features.csv"
joined_path = (
    PROCESSED_DIR
    / "join_train_backtest"
    / "training_features_joined_market.csv"
)

if not training_path.exists():
    raise FileNotFoundError("Missing data/processed/training_features.csv")

if not joined_path.exists():
    raise FileNotFoundError(
        "Missing join_train_backtest/training_features_joined_market.csv. "
        "Run 03 first."
    )

training = pd.read_csv(training_path)
joined = pd.read_csv(joined_path)

training["date"] = pd.to_datetime(training["date"], errors="coerce")
joined["date"] = pd.to_datetime(joined["date"], errors="coerce")

training["home_team"] = training["home_team"].map(norm_team)
training["away_team"] = training["away_team"].map(norm_team)
joined["home_team"] = joined["home_team"].map(norm_team)
joined["away_team"] = joined["away_team"].map(norm_team)

training = training.dropna(subset=["date", "outcome"]).copy()
joined = joined.dropna(subset=["date", "outcome"]).copy()

training["outcome"] = training["outcome"].astype(int)
joined["outcome"] = joined["outcome"].astype(int)

print("training:", training.shape)
print("joined:", joined.shape)

display(training.head())
display(joined.head())


In [ ]:
# Cell 3. Select football features.

exclude = {
    "date",
    "home_team",
    "away_team",
    "tournament",
    "outcome",
    "home_score",
    "away_score",
}

football_features = [
    col for col in training.columns
    if col not in exclude
    and pd.api.types.is_numeric_dtype(training[col])
]

pd.Series(football_features).to_csv(
    OUT_DIR / "football_feature_columns.csv",
    index=False,
)

print("Football features:", len(football_features))
print(football_features)


In [ ]:
# Cell 4. Walk-forward football predictions for joined market matches.

# We train one model per match date.
# For speed, train once per unique date, then predict all joined matches on that date.

MIN_TRAIN_ROWS = 1000
C_VALUE = 0.5

pred_rows = []

unique_dates = sorted(joined["date"].dropna().unique())

for date in unique_dates:
    train_df = training[training["date"] < date].copy()
    test_df = joined[joined["date"] == date].copy()

    if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
        continue

    model = make_logit_pipeline(C=C_VALUE)

    model.fit(
        train_df[football_features],
        train_df["outcome"],
    )

    raw = model.predict_proba(test_df[football_features])

    proba = np.zeros((len(test_df), 3))

    for i, cls in enumerate(model.named_steps["model"].classes_):
        proba[:, int(cls)] = raw[:, i]

    for idx, (_, row) in enumerate(test_df.iterrows()):
        pred_rows.append({
            "date": row["date"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "outcome": int(row["outcome"]),
            "football_p_away": float(proba[idx, 0]),
            "football_p_draw": float(proba[idx, 1]),
            "football_p_home": float(proba[idx, 2]),
            "n_train_rows": int(len(train_df)),
        })

football_predictions = pd.DataFrame(pred_rows)

football_predictions.to_csv(
    OUT_DIR / "walkforward_football_predictions.csv",
    index=False,
)

print("walkforward predictions:", football_predictions.shape)
display(football_predictions.head())


In [ ]:
# Cell 5. Join football predictions back to market rows.

eval_df = joined.merge(
    football_predictions,
    on=["date", "home_team", "away_team", "outcome"],
    how="inner",
)

eval_df.to_csv(
    OUT_DIR / "walkforward_eval_dataset.csv",
    index=False,
)

print("eval_df:", eval_df.shape)
display(eval_df.head())

if len(eval_df) == 0:
    raise ValueError("No walk-forward predictions joined.")


In [ ]:
# Cell 6. Evaluate market-only and football-only.

metric_rows = []

y = eval_df["outcome"].astype(int).to_numpy()

football_proba = eval_df[
    ["football_p_away", "football_p_draw", "football_p_home"]
].to_numpy()

metric_rows.append(
    evaluate_probs("walkforward_football_only", y, football_proba)
)

for snap in ["T_minus_24h", "T_minus_3h", "T_minus_1h"]:
    cols = [
        f"{snap}_market_p_away_devig",
        f"{snap}_market_p_draw_devig",
        f"{snap}_market_p_home_devig",
    ]

    if all(c in eval_df.columns for c in cols):
        tmp = eval_df.dropna(subset=cols).copy()

        if len(tmp) > 0:
            metric_rows.append(
                evaluate_probs(
                    f"market_only_{snap}",
                    tmp["outcome"].to_numpy(),
                    tmp[cols].to_numpy(),
                )
            )

baseline_metrics = pd.DataFrame(metric_rows)

baseline_metrics.to_csv(
    OUT_DIR / "baseline_metrics.csv",
    index=False,
)

display(baseline_metrics)


In [ ]:
# Cell 7. Simple blend alpha search.

# p_blend = alpha * football + (1 - alpha) * market_T24.
# alpha = 0 means pure market.
# alpha = 1 means pure football.

market_cols_24 = [
    "T_minus_24h_market_p_away_devig",
    "T_minus_24h_market_p_draw_devig",
    "T_minus_24h_market_p_home_devig",
]

if not all(c in eval_df.columns for c in market_cols_24):
    raise ValueError("T_minus_24h market columns missing.")

blend_df = eval_df.dropna(subset=market_cols_24).copy()

y = blend_df["outcome"].astype(int).to_numpy()

market_p = blend_df[market_cols_24].to_numpy()
football_p = blend_df[
    ["football_p_away", "football_p_draw", "football_p_home"]
].to_numpy()

blend_rows = []

for alpha in np.round(np.linspace(0.0, 1.0, 21), 2):
    p = alpha * football_p + (1.0 - alpha) * market_p
    p = p / p.sum(axis=1, keepdims=True)

    row = evaluate_probs(
        f"blend_alpha_{alpha:.2f}",
        y,
        p,
    )

    row["alpha_football"] = float(alpha)
    row["alpha_market"] = float(1.0 - alpha)

    blend_rows.append(row)

blend_metrics = pd.DataFrame(blend_rows).sort_values("log_loss")

best_alpha = float(blend_metrics.iloc[0]["alpha_football"])

blend_metrics.to_csv(
    OUT_DIR / "blend_alpha_search_metrics.csv",
    index=False,
)

display(blend_metrics.head(10))

print("Best alpha football:", best_alpha)


In [ ]:
# Cell 8. Build edge backtest with best blend.

p_blend = best_alpha * football_p + (1.0 - best_alpha) * market_p
p_blend = p_blend / p_blend.sum(axis=1, keepdims=True)

for i, name in enumerate(["away", "draw", "home"]):
    blend_df[f"blend_p_{name}"] = p_blend[:, i]

edge_rows = []

for _, row in blend_df.iterrows():
    for outcome_name, class_id in [
        ("away", 0),
        ("draw", 1),
        ("home", 2),
    ]:
        model_p = row[f"blend_p_{outcome_name}"]
        market_p = row[f"T_minus_24h_market_p_{outcome_name}_devig"]
        odds_24h = row.get(f"T_minus_24h_best_{outcome_name}_odds", np.nan)
        odds_1h = row.get(f"T_minus_1h_best_{outcome_name}_odds", np.nan)

        if pd.isna(model_p) or pd.isna(market_p) or pd.isna(odds_24h):
            continue

        edge = model_p - market_p
        ev = model_p * odds_24h - 1.0
        won = int(int(row["outcome"]) == class_id)

        profit_1usd = (
            odds_24h - 1.0
            if won
            else -1.0
        )

        clv_odds = (
            odds_24h / odds_1h - 1.0
            if pd.notna(odds_1h) and odds_1h > 0
            else np.nan
        )

        edge_rows.append({
            "date": row["date"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "outcome": int(row["outcome"]),
            "selection": outcome_name,
            "selection_class": class_id,
            "model_p_blend": model_p,
            "market_p_24h": market_p,
            "edge": edge,
            "best_odds_24h": odds_24h,
            "best_odds_1h": odds_1h,
            "model_ev": ev,
            "selection_won": won,
            "profit_1usd": profit_1usd,
            "clv_proxy_odds": clv_odds,
        })

edge_backtest = pd.DataFrame(edge_rows)

edge_backtest.to_csv(
    OUT_DIR / "edge_backtest_all_selections.csv",
    index=False,
)

display(edge_backtest.head(30))
print("edge rows:", len(edge_backtest))


In [ ]:
# Cell 9. Edge threshold report.

threshold_rows = []

for min_edge in [0.02, 0.03, 0.05, 0.07, 0.10]:
    for min_ev in [0.00, 0.03, 0.05, 0.10]:
        picks = edge_backtest[
            (edge_backtest["edge"] >= min_edge)
            & (edge_backtest["model_ev"] >= min_ev)
        ].copy()

        if len(picks) == 0:
            continue

        threshold_rows.append({
            "min_edge": min_edge,
            "min_ev": min_ev,
            "n_picks": len(picks),
            "hit_rate": float(picks["selection_won"].mean()),
            "roi_1usd_flat": float(picks["profit_1usd"].mean()),
            "total_profit_1usd_flat": float(picks["profit_1usd"].sum()),
            "avg_clv_proxy_odds": float(picks["clv_proxy_odds"].mean()),
            "median_clv_proxy_odds": float(picks["clv_proxy_odds"].median()),
            "avg_model_ev": float(picks["model_ev"].mean()),
            "avg_edge": float(picks["edge"].mean()),
        })

threshold_report = pd.DataFrame(threshold_rows)

if len(threshold_report) > 0:
    threshold_report = threshold_report.sort_values(
        ["roi_1usd_flat", "n_picks"],
        ascending=[False, False],
    )

threshold_report.to_csv(
    OUT_DIR / "edge_threshold_report.csv",
    index=False,
)

display(threshold_report)


In [ ]:
# Cell 10. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "training_rows": int(len(training)),
    "joined_rows": int(len(joined)),
    "eval_rows": int(len(eval_df)),
    "football_features": int(len(football_features)),
    "best_alpha_football": float(best_alpha),
    "edge_rows": int(len(edge_backtest)),
    "threshold_rows": int(len(threshold_report)),
    "real_money_allowed": False,
    "note": (
        "This is a paper backtest only. "
        "Do not use for real-money betting."
    ),
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "07_walkforward_market_backtest_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
